# 01 | Data exploration and understanding

Exploration of raw events, sessions, HTTP requests, user accounts, and cancellations.

**Source:** `archive/Project_fin_original.ipynb`. The original computations, structure, and results have been preserved. The notebook text has been translated into English; standalone setup and local checkpoints support reproducible execution.


In [ ]:
# Standalone setup: no src/ folder required.
# Run the notebooks in numerical order (02 -> 07); 01 is optional exploration.
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd

working_dir = Path.cwd().resolve()
# Works when VS Code/Jupyter starts in the repository root or notebooks/.
if working_dir.name == "notebooks":
    project_root = working_dir.parent
elif (working_dir / "notebooks").is_dir():
    project_root = working_dir
elif (working_dir.parent / "notebooks").is_dir():
    project_root = working_dir.parent
else:
    project_root = working_dir

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

HORIZON = pd.Timedelta(days=10)
CUTOFFS = pd.date_range(start="2018-10-11", end="2018-11-10", freq="10D")
FEATURES_12 = [
    "n_sessions_10d", "n_active_days_10d", "n_songs_10d",
    "n_likes_10d", "n_dislikes_10d", "n_errors_10d",
    "n_playlist_adds_10d", "total_song_length_10d", "n_artists_10d",
    "downgraded_10d", "is_paid", "songs_trend_10d"
]
EVENT_COLS = [
    "userId", "time", "registration", "page", "sessionId", "level",
    "gender", "length", "artist", "song", "status", "method",
    "userAgent", "auth", "firstName", "lastName", "itemInSession", "location"
]

def raw_path(kind):
    if kind not in ("train", "test"):
        raise ValueError("kind must be 'train' or 'test'")
    filename = f"{kind}.parquet"
    candidates = [
        project_root / filename,
        project_root / "data" / "raw" / filename,
        project_root / "notebooks" / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Missing {filename}. Put it in the repository root or data/raw/. "
        f"Current root: {project_root}"
    )

def load_events(kind):
    frame = pd.read_parquet(raw_path(kind), columns=EVENT_COLS)
    frame["userId"] = frame["userId"].astype(str)
    frame["time"] = pd.to_datetime(frame["time"])
    frame["registration"] = pd.to_datetime(frame["registration"])
    return frame

def data_path(filename):
    folder = project_root / "data" / "processed"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / filename

def artifact_path(filename):
    folder = project_root / "artifacts"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / filename

def save_cv_state(dev_data, holdout_data, cv_splits):
    dev_data.to_parquet(data_path("dev_data.parquet"), index=False)
    holdout_data.to_parquet(data_path("holdout_data.parquet"), index=False)
    fold_arrays = {}
    for i, (idx_train, idx_valid) in enumerate(cv_splits):
        fold_arrays[f"train_{i}"] = np.asarray(idx_train, dtype=np.int64)
        fold_arrays[f"valid_{i}"] = np.asarray(idx_valid, dtype=np.int64)
    np.savez_compressed(data_path("cv_splits.npz"), **fold_arrays)

def load_cv_state():
    dev = pd.read_parquet(data_path("dev_data.parquet"))
    holdout = pd.read_parquet(data_path("holdout_data.parquet"))
    with np.load(data_path("cv_splits.npz")) as saved:
        n = sum(name.startswith("train_") for name in saved.files)
        splits = [
            (saved[f"train_{i}"].copy(), saved[f"valid_{i}"].copy())
            for i in range(n)
        ]
    return dev, holdout, splits

print("Project folder:", project_root)




In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from catboost import CatBoostClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    VotingClassifier    ### Combine classifiers predictions
)
from sklearn.linear_model import LogisticRegression

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# 1) DATA EXPLORATION

In [2]:
df = pd.read_parquet(raw_path("train"))
print(df.shape)
df.info()

(17499636, 19)
<class 'pandas.core.frame.DataFrame'>
Index: 17499636 entries, 0 to 25661583
Data columns (total 19 columns):
 #   Column         Dtype         
---  ------         -----         
 0   status         int64         
 1   gender         object        
 2   firstName      object        
 3   level          object        
 4   lastName       object        
 5   userId         object        
 6   ts             int64         
 7   auth           object        
 8   page           object        
 9   sessionId      int64         
 10  location       object        
 11  itemInSession  int64         
 12  userAgent      object        
 13  method         object        
 14  length         float64       
 15  song           object        
 16  artist         object        
 17  time           datetime64[us]
 18  registration   datetime64[us]
dtypes: datetime64[us](2), float64(1), int64(4), object(12)
memory usage: 2.6+ GB


In [3]:
print(df.status.value_counts())
df.head(30)


status
200    16020693
307     1461649
404       17294
Name: count, dtype: int64


,status,gender,firstName,level,lastName,userId,ts,auth,page,sessionId,location,itemInSession,userAgent,method,length,song,artist,time,registration
0,200,M,Shlok,paid,Johnson,1749042,1538352001000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",278,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,524.32934,Ich mache einen Spiegel - Dream Part 4,Popol Vuh,2018-10-01 00:00:01,2018-08-08 13:22:21
992,200,M,Shlok,paid,Johnson,1749042,1538352525000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",279,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,178.02404,Monster (Album Version),Skillet,2018-10-01 00:08:45,2018-08-08 13:22:21
1360,200,M,Shlok,paid,Johnson,1749042,1538352703000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",280,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,232.61995,Seven Nation Army,The White Stripes,2018-10-01 00:11:43,2018-08-08 13:22:21
1825,200,M,Shlok,paid,Johnson,1749042,1538352935000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",281,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,265.50812,Under The Bridge (Album Version),Red Hot Chili Peppers,2018-10-01 00:15:35,2018-08-08 13:22:21
2366,200,M,Shlok,paid,Johnson,1749042,1538353200000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",282,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,471.69261,Circlesong 6,Bobby McFerrin,2018-10-01 00:20:00,2018-08-08 13:22:21
3271,200,M,Shlok,paid,Johnson,1749042,1538353671000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",283,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,266.86649,Who Can Compare,Foolish Things,2018-10-01 00:27:51,2018-08-08 13:22:21
3802,200,M,Shlok,paid,Johnson,1749042,1538353937000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",284,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,1400.65914,Angel Dust,Gil Scott Heron,2018-10-01 00:32:17,2018-08-08 13:22:21
6585,200,M,Shlok,paid,Johnson,1749042,1538355337000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",285,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,186.98404,Sweet And Dandy,Toots & The Maytals,2018-10-01 00:55:37,2018-08-08 13:22:21
6675,200,M,Shlok,paid,Johnson,1749042,1538355388000,Logged In,Downgrade,22683,"Dallas-Fort Worth-Arlington, TX",286,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",GET,NaN,None,None,2018-10-01 00:56:28,2018-08-08 13:22:21
6961,200,M,Shlok,paid,Johnson,1749042,1538355523000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",287,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,306.05016,On The Moon,Peter Cincotti,2018-10-01 00:58:43,2018-08-08 13:22:21


In [4]:
X= df[df['userId'] == '1749042']
X[X['status'] != 200]

,status,gender,firstName,level,lastName,userId,ts,auth,page,sessionId,location,itemInSession,userAgent,method,length,song,artist,time,registration
24330,307,M,Shlok,paid,Johnson,1749042,1538363406000,Logged In,Thumbs Up,22683,"Dallas-Fort Worth-Arlington, TX",324,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-01 03:10:06,2018-08-08 13:22:21
391536,307,M,Shlok,paid,Johnson,1749042,1538467226000,Logged In,Add Friend,25570,"Dallas-Fort Worth-Arlington, TX",4,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-02 08:00:26,2018-08-08 13:22:21
883667,307,M,Shlok,paid,Johnson,1749042,1538591742000,Logged In,Thumbs Down,31304,"Dallas-Fort Worth-Arlington, TX",6,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-03 18:35:42,2018-08-08 13:22:21
893741,307,M,Shlok,paid,Johnson,1749042,1538593847000,Logged In,Thumbs Down,31304,"Dallas-Fort Worth-Arlington, TX",18,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-03 19:10:47,2018-08-08 13:22:21
916990,307,M,Shlok,paid,Johnson,1749042,1538598939000,Logged In,Thumbs Up,31304,"Dallas-Fort Worth-Arlington, TX",37,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-03 20:35:39,2018-08-08 13:22:21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6641195,307,M,Shlok,paid,Johnson,1749042,1540076041000,Logged In,Thumbs Up,100362,"Dallas-Fort Worth-Arlington, TX",16,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-20 22:54:01,2018-08-08 13:22:21
6651435,307,M,Shlok,paid,Johnson,1749042,1540080154000,Logged In,Save Settings,100362,"Dallas-Fort Worth-Arlington, TX",37,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-21 00:02:34,2018-08-08 13:22:21
6658537,307,M,Shlok,paid,Johnson,1749042,1540082944000,Logged In,Thumbs Up,100362,"Dallas-Fort Worth-Arlington, TX",51,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-21 00:49:04,2018-08-08 13:22:21
6660411,307,M,Shlok,paid,Johnson,1749042,1540083704000,Logged In,Logout,100362,"Dallas-Fort Worth-Arlington, TX",56,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,NaN,None,None,2018-10-21 01:01:44,2018-08-08 13:22:21


In [5]:
COLS = [
    "userId", "time", "registration",
    "page", "sessionId", "level",
    "gender", "length", "artist",
    "song", "status", "method",
    "userAgent", "auth", "firstName", 
    "lastName", "itemInSession", "location"
]

train = pd.read_parquet(raw_path("train"), columns=COLS)
test = pd.read_parquet(raw_path("test"), columns=COLS)

for data in (train, test):
    data["userId"] = data["userId"].astype(str)
    data["time"] = pd.to_datetime(data["time"])
    data["registration"] = pd.to_datetime(data["registration"])

# 7. Sanity checks
print("Train :", train.shape, "dates:", train.time.min(), train.time.max())
print("Test  :", test.shape, "dates:", test.time.min(), test.time.max())
print("Common Users :", len(set(train.userId) & set(test.userId)))
print(
    "Churn present in test:",
    test.loc[
        test.page.eq("Cancellation Confirmation"), "userId"
    ].nunique())

print("Auth states :")
print(train["auth"].value_counts(dropna=False)) ### No need corresp to page which has more info

print("\nNumber of different User Agents  :")
print(train["userAgent"].nunique())

print("\nMost frequent user agents:")
print(train["userAgent"].value_counts().head(10))

print("\nUsers with >1 User Agent :")
print(
    (train.groupby("userId")["userAgent"].nunique() > 1).sum()
)

Train : (17499636, 18) dates: 2018-10-01 00:00:01 2018-11-20 00:00:00
Test  : (4393179, 18) dates: 2018-10-01 00:00:06 2018-11-20 00:00:00
Common Users : 0
Churn present in test: 0
Auth states :
auth
Logged In    17495365
Cancelled        4271
Name: count, dtype: int64

Number of different User Agents  :
85

Most frequent user agents:
userAgent
"Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.143 Safari/537.36"               1605877
Mozilla/5.0 (Windows NT 6.1; WOW64; rv:31.0) Gecko/20100101 Firefox/31.0                                                      1365774
"Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.125 Safari/537.36"               1063498
"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_4) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.143 Safari/537.36"    1051132
"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_4) AppleWebKit/537.78.2 (KHTML, like Gecko) Version/7.0.6 Safari/537.78.

In [6]:
# Number of distinct values per user
users_info = train.groupby("userId").agg(
    n_firstnames=("firstName", "nunique"),
    n_lastnames=("lastName", "nunique"),
    n_agents=("userAgent", "nunique"),
    n_locations=("location", "nunique")
)

print("UserId with >1 first name :",
      (users_info["n_firstnames"] > 1).sum())

print("UserId with >1 last name :",
      (users_info["n_lastnames"] > 1).sum())

print("UserId with >1 location :",
      (users_info["n_locations"] > 1).sum())

# How many distinct accounts share each user agent?
print("\nNumber of users per user agent:")
print(
    train.groupby("userAgent")["userId"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

# Explore locations
print("\nMost frequent locations:")
print(train["location"].value_counts().head(10))

# Explore itemInSession
print("\nitemInSession Distribution:")
print(train["itemInSession"].describe(
    percentiles=[0.5, 0.9, 0.99]
))


UserId with >1 first name : 0
UserId with >1 last name : 0
UserId with >1 location : 0

Number of users per user agent:
userAgent
"Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.143 Safari/537.36"               1712
Mozilla/5.0 (Windows NT 6.1; WOW64; rv:31.0) Gecko/20100101 Firefox/31.0                                                      1437
"Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.125 Safari/537.36"               1205
"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_4) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.143 Safari/537.36"    1139
"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_4) AppleWebKit/537.78.2 (KHTML, like Gecko) Version/7.0.6 Safari/537.78.2"       1046
"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_4) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/36.0.1985.125 Safari/537.36"     930
Mozilla/5.0 (Macintosh; Intel Mac OS X 10.9; rv:31.0) Gecko/20100101 Firefox/31.0   

In [7]:
print("HTTP status code distribution:")
print(train["status"].value_counts(dropna=False))

print("\nHTTP method distribution:")
print(train["method"].value_counts(dropna=False))

print("\nHTTP status codes by page:")
print(pd.crosstab(train["page"], train["status"]))

HTTP status code distribution:
status
200    16020693
307     1461649
404       17294
Name: count, dtype: int64

HTTP method distribution:
method
PUT    16162688
GET     1336948
Name: count, dtype: int64

HTTP status codes by page:
status                          200     307    404
page                                              
About                         33117       0      0
Add Friend                        0  262147      0
Add to Playlist              409606       0      0
Cancel                            0    4271      0
Cancellation Confirmation      4271       0      0
Downgrade                    124248       0      0
Error                             0       0  17294
Help                          89035       0      0
Home                         645259       0      0
Logout                            0  204700      0
NextSong                   14291433       0      0
Roll Advert                  284837       0      0
Save Settings                     0   20370      0
Set

In [8]:
print("\nHTTP Methods by page type :")
print(pd.crosstab(train["page"], train["method"]))


HTTP Methods by page type :
method                        GET       PUT
page                                       
About                       33117         0
Add Friend                      0    262147
Add to Playlist                 0    409606
Cancel                          0      4271
Cancellation Confirmation    4271         0
Downgrade                  124248         0
Error                       17294         0
Help                        89035         0
Home                       645259         0
Logout                          0    204700
NextSong                        0  14291433
Roll Advert                284837         0
Save Settings                   0     20370
Settings                   101191         0
Submit Downgrade                0      4425
Submit Upgrade                  0     11381
Thumbs Down                     0    164964
Thumbs Up                       0    789391
Upgrade                     37696         0


In [9]:
# Number of events per session
sessions = (
    train.groupby(["userId", "sessionId"])
    .size()
    .reset_index(name="nb_events")
)

# Session-size distribution
print(sessions["nb_events"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]
))

count    208358.000000
mean         83.988309
std         101.473892
min           1.000000
25%          19.000000
50%          49.000000
75%         109.000000
90%         207.000000
99%         484.000000
max        1393.000000
Name: nb_events, dtype: float64


In [10]:
sessions_dates = train.groupby(
    ["userId", "sessionId"]
)["time"].agg(["min", "max"])

# Sessions starting and ending on different calendar days
sessions_multi_days = sessions_dates[
    sessions_dates["min"].dt.date
    != sessions_dates["max"].dt.date
]

print("Total sessions:", len(sessions_dates))
print("Sessions crossing midnight:", len(sessions_multi_days))

print("\nExamples:")
print(sessions_multi_days.head())

Total sessions: 208358
Sessions crossing midnight: 39416

Examples:
                                  min                 max
userId  sessionId                                        
1000025 31688     2018-10-02 18:12:22 2018-10-03 21:31:44
        60612     2018-10-09 23:27:02 2018-10-10 09:26:10
        69552     2018-10-11 18:48:52 2018-10-12 00:21:00
        85743     2018-10-15 19:58:41 2018-10-16 01:40:29
        86823     2018-10-16 22:36:23 2018-10-17 07:39:06


In [11]:
# Session duration in minutes
sessions_dates["duration_min"] = (
    sessions_dates["max"] - sessions_dates["min"]
).dt.total_seconds() / 60

print(sessions_dates["duration_min"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]
))

count    208358.000000
mean        288.780987
std         364.305263
min           0.000000
25%          57.416667
50%         159.800000
75%         375.929167
90%         726.916667
99%        1734.817000
max        5790.583333
Name: duration_min, dtype: float64


In [12]:
# Find the longest session
user, session = sessions_dates["duration_min"].idxmax()

# Retrieve the events in that session
long_session = train[
    (train["userId"] == user) &
    (train["sessionId"] == session)
].sort_values("time").copy()

# Elapsed time between consecutive events
long_session["gap_min"] = (
    long_session["time"].diff().dt.total_seconds() / 60
)

print("User:", user)
print("Session :", session)
print("Number of events:", len(long_session))

print("\nTop 10 longest gaps between events:")
print(
    long_session.nlargest(10, "gap_min")[
        ["time", "page", "itemInSession", "gap_min"]
    ]
)

User: 1642682
Session : 89129
Number of events: 1329

Top 10 longest gaps between events:
                       time             page  itemInSession    gap_min
5766698 2018-10-18 11:29:41             Home            505  48.483333
6611006 2018-10-20 19:45:36        Downgrade           1308  46.466667
6268953 2018-10-19 17:28:12         NextSong            929  36.300000
5293683 2018-10-17 05:31:35             Home             97  35.766667
5319605 2018-10-17 07:27:03  Add to Playlist            121  31.700000
5259286 2018-10-17 03:00:55             Home             58  29.416667
6389370 2018-10-19 23:53:18             Home           1034  29.400000
6543120 2018-10-20 13:09:26  Add to Playlist           1229  29.083333
5326155 2018-10-17 07:55:45         NextSong            122  28.700000
6132828 2018-10-19 09:31:56             Home            818  28.650000


In [13]:

# Last event in each session
last_events = train.groupby(
    ["userId", "sessionId"]
)["itemInSession"].max()

# Last logout in each relevant session
logout_events = train[
    train["page"] == "Logout"
].groupby(
    ["userId", "sessionId"]
)["itemInSession"].max()

# Sessions containing at least one logout
n_logout = len(logout_events)

# Sessions with an event after their last logout
n_after_logout = (
    logout_events < last_events.loc[logout_events.index]
).sum()

print("Total sessions:", len(last_events))
print("Sessions with a logout:", n_logout)
print("Sessions without a logout:", len(last_events) - n_logout)
print("Sessions with events after the last logout:", n_after_logout)


Total sessions: 208358
Sessions with a logout: 111545
Sessions without a logout: 96813
Sessions with events after the last logout: 78236


In [14]:

# Sessions where events follow the last logout
mask = logout_events < last_events.loc[logout_events.index]

# Select an example
user, session = mask[mask].index[0]

# Retrieve events from that session
example = train[
    (train["userId"] == user) &
    (train["sessionId"] == session)
].sort_values("itemInSession").reset_index(drop=True)

# Position of the last logout
logout_pos = example.index[
    example["page"] == "Logout"
][-1]

# Inspect events before and after the logout
print(
    example.iloc[
        max(0, logout_pos - 3):logout_pos + 6
    ][["time", "page", "itemInSession"]]
)


                   time      page  itemInSession
408 2018-10-03 17:44:21  NextSong            413
409 2018-10-03 17:49:24  NextSong            414
410 2018-10-03 17:53:38  NextSong            415
411 2018-10-03 17:53:39    Logout            416
412 2018-10-03 17:54:23      Home            420
413 2018-10-03 17:56:16  NextSong            421
414 2018-10-03 18:00:25  NextSong            422
415 2018-10-03 18:03:41  NextSong            423
416 2018-10-03 18:07:00  NextSong            424


In [15]:
# Cancellation date for each user
cancellations = (
    train[train["page"] == "Cancellation Confirmation"]
    .groupby("userId")["time"]
    .min()
)

# Events generated by users who cancelled
churn_events = train[
    train["userId"].isin(cancellations.index)
].copy()

# Attach the cancellation date to each event
churn_events["cancel_date"] = (
    churn_events["userId"].map(cancellations)
)

# Keep events occurring after cancellation
after_cancel = churn_events[
    churn_events["time"] > churn_events["cancel_date"]
]

print("Users with activity after cancellation:",
      after_cancel["userId"].nunique())

print("\nEvents after cancellation:")
print(after_cancel["page"].value_counts())


Users with activity after cancellation: 0

Events after cancellation:
Series([], Name: count, dtype: int64)
